# Size, speed and memory
One row per model: the teacher, and each student in its best arm (lowest mean test RMSE in `results/distill_runs.csv`), as saved by `python -m src.build_artifacts`. Every model is one `joblib` file in `artifacts/`, so sizes on disk are comparable.

**Protocol.**
- **Threads:** one thread for everything (`threadpool_limits(1)`, and `n_jobs=1` for the teacher's LightGBM and forest). Memory and latency use the same setting.
- **Single-row latency:** one `predict` call on one row at a time, timed with `time.perf_counter_ns`. 1,000 warm-up calls are thrown away, then 5,000 timed calls cycle through real test rows. Reported as p50 and p99. Building the input row is not timed.
- **Full call vs. model only:** the full call is what a caller sees (`model.predict(row)`), including Python and scikit-learn input checks. For small models that overhead dominates, so the model-only time strips it where possible. For the MLPs that is the whole forward pass in plain numpy. For the tree it is `tree_.predict`. For the teacher it is the two libraries' own predict calls (LightGBM's booster and the forest), which still carry each library's own overhead.
- **Batch throughput:** rows per second predicting a 200,000-row batch in one call, best of 3.
- **Memory:** `python -m src.measure_memory <artifact>` in a fresh process per model. It reports resident memory before and after loading the artifact, and the peak while predicting the 200,000-row batch.

In [1]:
import json
import subprocess
import sys
import time

import joblib
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

from src.bench import count_parameters, mlp_forward, pin_threads, single_row_latency
from src.build_artifacts import ARTIFACT_DIR, artifact_name, best_arms
from src.lib import RESULTS_DIR, ROOT, load_data, load_split

hw = subprocess.run(["system_profiler", "SPHardwareDataType"], capture_output=True, text=True).stdout
print("\n".join(l.strip() for l in hw.splitlines() if any(k in l for k in ("Model Name", "Model Identifier", "Chip", "Total Number of Cores", "Memory"))))
print(subprocess.run(["pmset", "-g", "batt"], capture_output=True, text=True).stdout.splitlines()[0])

Model Name: MacBook Pro
Model Identifier: Mac17,9
Chip: Apple M5 Pro
Total Number of Cores: 18 (6 Super and 12 Performance)
Memory: 64 GB
Now drawing from 'AC Power'


## Load the artifacts and the test rows

In [2]:
runs = pd.read_csv(RESULTS_DIR / "distill_runs.csv")
arm_of = best_arms(runs)
names = ["Teacher (LightGBM + forest)"] + list(arm_of)
paths = {n: ARTIFACT_DIR / ("teacher.joblib" if n.startswith("Teacher") else f"{artifact_name(n)}.joblib") for n in names}
arm_label = {"Teacher (LightGBM + forest)": "-", **arm_of}

X, _ = load_data()
test_idx, _folds = load_split()
X_test = X.to_numpy()[test_idx]
rows = [np.ascontiguousarray(X_test[i : i + 1]) for i in range(len(X_test))]
rows32 = [r.astype(np.float32) for r in rows]  # the tree's own predict wants float32
batch = np.tile(X_test, (200_000 // len(X_test) + 1, 1))[:200_000]
pd.Series({n: arm_label[n] for n in names}, name="arm used")

Teacher (LightGBM + forest)                          -
MLP 2x32                       3 distilled + synthetic
MLP 2x128                      3 distilled + synthetic
Tree (depth 6)                             2 distilled
Name: arm used, dtype: str

## Latency and throughput

In [3]:
def model_only_call(name, model):
    """(function, rows) that time just the model, without wrapper and input-check overhead."""
    if name.startswith("Teacher"):
        booster, forest = model.fitted_[0].model_.booster_, model.fitted_[1]
        return (lambda r: 0.5 * (booster.predict(r, num_threads=1)[0] + forest.predict(r)[0])), rows
    if name.startswith("MLP"):
        return mlp_forward(model), rows
    return (lambda r: model.tree_.predict(r)[0, 0]), rows32


results = {}
for name in names:
    model = pin_threads(joblib.load(paths[name]))
    with threadpool_limits(limits=1):
        full = single_row_latency(model.predict, rows)
        fn, fn_rows = model_only_call(name, model)
        only = single_row_latency(fn, fn_rows)
        model.predict(batch)  # warm up
        batch_times = []
        for _ in range(3):
            t0 = time.perf_counter()
            model.predict(batch)
            batch_times.append(time.perf_counter() - t0)
    results[name] = {
        "arm": arm_label[name],
        "size_mb": paths[name].stat().st_size / 1024**2,
        "parameters": count_parameters(model),
        "p50_us": full["p50_us"],
        "p99_us": full["p99_us"],
        "model_only_p50_us": only["p50_us"],
        "model_only_p99_us": only["p99_us"],
        "batch_rows_per_s": len(batch) / min(batch_times),
    }
    print(name, "done", flush=True)

Teacher (LightGBM + forest) done


MLP 2x32 done


MLP 2x128 done


Tree (depth 6) done


## Memory, one fresh process per model

In [4]:
for name in names:
    out = subprocess.run(
        [sys.executable, "-m", "src.measure_memory", str(paths[name])],
        capture_output=True, text=True, cwd=ROOT, check=True,
    ).stdout.strip().splitlines()[-1]
    mem = json.loads(out)
    results[name].update(
        {
            "rss_before_mb": mem["rss_before_mb"],
            "load_mb": mem["load_mb"],
            "peak_rss_during_predict_mb": mem["peak_rss_during_predict_mb"],
        }
    )
    print(name, "done", flush=True)

Teacher (LightGBM + forest) done


MLP 2x32 done


MLP 2x128 done


Tree (depth 6) done


## The table

In [5]:
table = pd.DataFrame(results).T
table.index.name = "model"
for col in table.columns[1:]:
    table[col] = table[col].astype(float)
table.to_csv(RESULTS_DIR / "size_speed_memory.csv")
table.round(2)

,arm,size_mb,parameters,p50_us,p99_us,model_only_p50_us,model_only_p99_us,batch_rows_per_s,rss_before_mb,load_mb,peak_rss_during_predict_mb
model,,,,,,,,,,,
Teacher (LightGBM + forest),-,417.74,6098767.0,5534.92,7080.02,5534.44,7053.66,3.101373e+04,196.42,857.30,1060.75
MLP 2x32,3 distilled + synthetic,0.04,1377.0,123.38,150.38,5.08,5.62,1.315407e+07,196.33,0.08,308.58
MLP 2x128,3 distilled + synthetic,0.42,17793.0,124.12,153.96,5.96,6.42,3.584235e+06,196.33,0.11,601.66
Tree (depth 6),2 distilled,0.01,127.0,23.42,68.67,1.00,1.46,1.222898e+08,196.33,0.11,204.83


### Findings
Measured on a MacBook Pro (Mac17,9), Apple M5 Pro, 18 cores, 64 GB, on AC power, one thread per model.

- **Size.** The teacher is a 418 MB file with 6.1 million tree nodes. The students are 0.01 to 0.42 MB: 1,377 weights for the 32-wide MLP, 17,793 for the 128-wide one and 127 nodes for the tree. All are one `joblib` file, so the sizes are comparable. The students are 1,000 to 40,000 times smaller.
- **Single-row latency (p50 / p99).** The teacher takes 5.5 ms / 7.1 ms per call. Both MLPs take about 124 microseconds / 150 to 154 microseconds, about 45 times faster, and the tree 23 / 69 microseconds. The teacher is slow because a single row still runs 300 forest trees plus the boosted model.
- **Most of a student's time is overhead, not the model.** For the MLPs the pure numpy forward pass takes 5 to 6 microseconds, against 124 for the full call, so about 95% is Python and scikit-learn input checks. For the tree it is 1.0 against 23. For the teacher the model-only time is the same as the full call (5.5 ms), because the work is in the trees themselves. A student served without that overhead would be about 1,000 times faster than the teacher, not 45.
- **Batch throughput.** The teacher manages 31,000 rows per second on one thread. The 128-wide MLP does 3.6 million (115 times faster), the 32-wide MLP 13 million (420 times) and the tree 122 million.
- **Memory.** Loading the teacher adds 857 MB and its peak during a 200,000-row batch is 1,061 MB. Loading a student adds 0.1 MB. A student's peak (205 to 602 MB, against a 196 MB baseline for the Python process) is mostly the batch's own intermediate arrays, which is why the wider MLP peaks higher than the narrow one.
- **Which student.** The 128-wide MLP kept the most of the teacher's gain (62%) and still costs 0.42 MB, 124 microseconds a call and 0.1 MB to load. The tree is the cheapest by far but kept none of the accuracy.